In [1]:
from pathlib import Path

import pandas as pd


def find_project_root() -> Path:
    """Resolve repo root whether the notebook cwd is project root or notebooks/Models/."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data" / "raw").is_dir():
            return candidate
    raise FileNotFoundError("Could not find project root containing data/raw/")


PROJECT = find_project_root()
RAW_ROOT = PROJECT / "data" / "raw"
HC_DIR = RAW_ROOT / "HC"   # healthy controls -> label 0
PT_DIR = RAW_ROOT / "PT"   # patients -> label 1


def scan_audio_folder(folder: Path, subgroup: str, label: int) -> list[dict]:
    if not folder.is_dir():
        raise FileNotFoundError(f"Missing audio folder: {folder}")

    rows = []
    for wav in sorted(folder.glob("*.wav")):
        rows.append(
            {
                "file_path": str(wav.resolve()),
                "file": wav.name,
                "file_stem": wav.stem,
                "subgroup": subgroup,
                "depressed": label,
            }
        )
    return rows


hc_rows = scan_audio_folder(HC_DIR, "HC", 0)
pt_rows = scan_audio_folder(PT_DIR, "PT", 1)
audio_df = pd.DataFrame(hc_rows + pt_rows)

# First token in filenames like 01_CF56_1 -> participant id for grouped splits later.
audio_df["participant_id"] = audio_df["file_stem"].str.split("_").str[0]

n_hc = int((audio_df["subgroup"] == "HC").sum())
n_pt = int((audio_df["subgroup"] == "PT").sum())

print(f"Project root: {PROJECT}")
print(f"HC dir: {HC_DIR} ({n_hc} .wav)")
print(f"PT dir: {PT_DIR} ({n_pt} .wav)")
print(f"Total recordings: {len(audio_df)}")
print(f"Labels — depressed=0 (HC): {(audio_df['depressed'] == 0).sum()} | depressed=1 (PT): {(audio_df['depressed'] == 1).sum()}")
print(f"Unique participants: {audio_df['participant_id'].nunique()}")

audio_df.head()



Project root: c:\Users\janku\Documents\KCL\Research Project\Research Project
HC dir: c:\Users\janku\Documents\KCL\Research Project\Research Project\data\raw\HC (54 .wav)
PT dir: c:\Users\janku\Documents\KCL\Research Project\Research Project\data\raw\PT (64 .wav)
Total recordings: 118
Labels — depressed=0 (HC): 54 | depressed=1 (PT): 64
Unique participants: 70


,file_path,file,file_stem,subgroup,depressed,participant_id
0,C:\Users\janku\Documents\KCL\Research Project\...,01_CF56_1.wav,01_CF56_1,HC,0,01
1,C:\Users\janku\Documents\KCL\Research Project\...,02_CM57_2.wav,02_CM57_2,HC,0,02
2,C:\Users\janku\Documents\KCL\Research Project\...,03_CF30_3.wav,03_CF30_3,HC,0,03
3,C:\Users\janku\Documents\KCL\Research Project\...,04_CF57_3.wav,04_CF57_3,HC,0,04
4,C:\Users\janku\Documents\KCL\Research Project\...,05_CF41_3.wav,05_CF41_3,HC,0,05


In [2]:
import gc
from functools import partial

import librosa
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import DataLoader, Dataset
from transformers import WhisperModel, WhisperProcessor

try:
    audio_df
except NameError as e:
    raise RuntimeError("Run the data-scan cell first (audio_df).") from e


# --------------------------------------------------
# Config
# --------------------------------------------------

WHISPER_ID = "openai/whisper-small.en"
TARGET_SR = 16_000
MAX_SECONDS = 30
EPOCHS = 8
BATCH_SIZE = 2
LEARNING_RATE = 1e-3
TEST_SIZE = 0.2
RANDOM_STATE = 42
FREEZE_ENCODER = True
FORCE_CPU = False

RESULTS_PATH = PROJECT / "results" / "metrics" / "kintsugi_health"
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

if not FORCE_CPU and torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print(f"Device: {DEVICE} | recordings: {len(audio_df)} | freeze encoder: {FREEZE_ENCODER}")


# --------------------------------------------------
# Preprocessing
# --------------------------------------------------

def load_audio(path) -> np.ndarray:
    wav, _ = librosa.load(str(path), sr=TARGET_SR, mono=True)
    max_len = TARGET_SR * MAX_SECONDS
    if len(wav) > max_len:
        wav = wav[:max_len]
    return wav.astype(np.float32)


processor = WhisperProcessor.from_pretrained(WHISPER_ID)


# --------------------------------------------------
# Model (DAM-style: Whisper encoder + MLP head)
# --------------------------------------------------

class DAMLikeModel(nn.Module):
    """Whisper encoder + trainable head. Trained on HC/PT depressed label (binary)."""

    def __init__(self, freeze_encoder: bool = True) -> None:
        super().__init__()
        self.whisper = WhisperModel.from_pretrained(WHISPER_ID, low_cpu_mem_usage=True)
        if freeze_encoder:
            for param in self.whisper.parameters():
                param.requires_grad = False

        hidden_size = self.whisper.config.d_model
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1),
        )

    def forward(self, input_features: torch.Tensor) -> torch.Tensor:
        encoder_out = self.whisper.encoder(input_features)
        pooled = encoder_out.last_hidden_state.mean(dim=1)
        return self.head(pooled).squeeze(-1)


# --------------------------------------------------
# Dataset
# --------------------------------------------------

class AudioDepressionDataset(Dataset):
    def __init__(self, paths: list[str], labels: np.ndarray) -> None:
        self.paths = paths
        self.labels = labels.astype(np.float32)

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int) -> tuple[np.ndarray, float]:
        return load_audio(self.paths[idx]), float(self.labels[idx])


def whisper_collate(batch: list[tuple[np.ndarray, float]], proc: WhisperProcessor):
    waves, labels = zip(*batch)
    feats = proc(list(waves), sampling_rate=TARGET_SR, padding=True, return_tensors="pt")
    y = torch.tensor(labels, dtype=torch.float32)
    return feats.input_features, y


def train_one_epoch(model, loader, optimizer, loss_fn) -> float:
    model.train()
    total_loss = 0.0
    n_batches = 0
    for input_features, y in loader:
        input_features = input_features.to(DEVICE)
        y = y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = model(input_features)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += float(loss.item())
        n_batches += 1
    return total_loss / max(n_batches, 1)


@torch.inference_mode()
def predict_probs(model, loader) -> tuple[np.ndarray, np.ndarray]:
    model.eval()
    all_probs: list[np.ndarray] = []
    all_labels: list[np.ndarray] = []
    for input_features, y in loader:
        input_features = input_features.to(DEVICE)
        logits = model(input_features)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(y.numpy())
    return np.concatenate(all_probs), np.concatenate(all_labels)


# --------------------------------------------------
# Grouped train / test split (by participant_id)
# --------------------------------------------------

df = audio_df.reset_index(drop=True)
groups = df["participant_id"].astype(str)
labels = df["depressed"].to_numpy(dtype=np.float32)
paths = df["file_path"].tolist()

splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(df, labels, groups=groups))

train_paths = [paths[i] for i in train_idx]
test_paths = [paths[i] for i in test_idx]
y_train = labels[train_idx]
y_test = labels[test_idx]

train_loader = DataLoader(
    AudioDepressionDataset(train_paths, y_train),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=partial(whisper_collate, proc=processor),
)
test_loader = DataLoader(
    AudioDepressionDataset(test_paths, y_test),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=partial(whisper_collate, proc=processor),
)

print(
    f"Train: {len(train_paths)} rows ({df.iloc[train_idx]['participant_id'].nunique()} participants) | "
    f"Test: {len(test_paths)} rows ({df.iloc[test_idx]['participant_id'].nunique()} participants)"
)


# --------------------------------------------------
# Train + evaluate
# --------------------------------------------------

model = DAMLikeModel(freeze_encoder=FREEZE_ENCODER).to(DEVICE)
optimizer = torch.optim.AdamW(
    (p for p in model.parameters() if p.requires_grad),
    lr=LEARNING_RATE,
)
loss_fn = nn.BCEWithLogitsLoss()

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn)
    print(f"Epoch {epoch}/{EPOCHS} train_loss={train_loss:.4f}")

test_probs, test_true = predict_probs(model, test_loader)
test_pred = (test_probs >= 0.5).astype(int)

metrics = {
    "model": "DAM-like Whisper (HC/PT)",
    "n_total": len(df),
    "n_train": len(train_paths),
    "n_test": len(test_paths),
    "n_participants": df["participant_id"].nunique(),
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "freeze_encoder": FREEZE_ENCODER,
    "accuracy": accuracy_score(test_true, test_pred),
    "f1": f1_score(test_true, test_pred, zero_division=0),
    "roc_auc": roc_auc_score(test_true, test_probs) if len(np.unique(test_true)) > 1 else np.nan,
}

summary_df = pd.DataFrame([metrics])
pred_df = df.iloc[test_idx][["file_path", "file", "file_stem", "subgroup", "participant_id", "depressed"]].copy()
pred_df["pred_prob"] = test_probs
pred_df["pred_label"] = test_pred

summary_path = RESULTS_PATH / "dam_whisper_hc_pt_summary.csv"
pred_path = RESULTS_PATH / "dam_whisper_hc_pt_test_predictions.csv"
summary_df.to_csv(summary_path, index=False)
pred_df.to_csv(pred_path, index=False)

print("\nHeld-out test metrics:")
print(summary_df.to_string(index=False))
print(f"\nSaved: {summary_path}")
print(f"Saved: {pred_path}")

# Quick smoke inference on the first training file
sample_path = train_paths[0]
sample_wav = load_audio(sample_path)
sample_inputs = processor(sample_wav, sampling_rate=TARGET_SR, return_tensors="pt")
with torch.inference_mode():
    sample_logit = model(sample_inputs.input_features.to(DEVICE))
    sample_prob = torch.sigmoid(sample_logit).item()
print(f"\nSmoke inference — {Path(sample_path).name}: P(depressed)={sample_prob:.3f}")

model.cpu()
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Device: cpu | recordings: 118 | freeze encoder: True


preprocessor_config.json: 0.00B [00:00, ?B/s]

c:\Users\janku\Documents\KCL\Research Project\Research Project\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\janku\.cache\huggingface\hub\models--openai--whisper-small.en. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/805 [00:00<?, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

Train: 92 rows (56 participants) | Test: 26 rows (14 participants)


model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Epoch 1/8 train_loss=0.7325
Epoch 2/8 train_loss=0.6371
Epoch 3/8 train_loss=0.5845
Epoch 4/8 train_loss=0.5287
Epoch 5/8 train_loss=0.4928
Epoch 6/8 train_loss=0.4706
Epoch 7/8 train_loss=0.3865
Epoch 8/8 train_loss=0.3836

Held-out test metrics:
                   model  n_total  n_train  n_test  n_participants  epochs  batch_size  learning_rate  freeze_encoder  accuracy       f1  roc_auc
DAM-like Whisper (HC/PT)      118       92      26              70       8           2          0.001            True  0.846154 0.833333 0.964286

Saved: c:\Users\janku\Documents\KCL\Research Project\Research Project\results\metrics\kintsugi_health\dam_whisper_hc_pt_summary.csv
Saved: c:\Users\janku\Documents\KCL\Research Project\Research Project\results\metrics\kintsugi_health\dam_whisper_hc_pt_test_predictions.csv

Smoke inference — 02_CM57_2.wav: P(depressed)=0.152
